# Ingesta USGS Earthquake API -> capa raw
Notebook de ingesta cruda (raw) para el catálogo sísmico de USGS (FDSN Event Web Service).
Independiente del pipeline de TVmaze: usa su propio esquema, volumen, widgets y variables.

In [0]:
%pip install pendulum

In [0]:

import json
from pathlib import Path
import pendulum
import requests

In [0]:
# Detecta el catálogo actual (suele ser 'workspace' o 'main')
catalogo_actual = spark.sql("select current_catalog()").first()[0]
catalogo = catalogo_actual
esquema = "raw_seismic"
volumen = "usgs_quakes"

In [0]:

# Crea esquema y volumen si no existen (nombres exclusivos de este pipeline)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalogo}.{esquema}.{volumen}")

In [0]:

# Elimina todos los archivos y subcarpetas dentro de la ruta de este pipeline
dbutils.fs.rm("/Volumes/workspace/raw_seismic/usgs_quakes", recurse=True)


In [0]:
dbutils.widgets.text("fecha_inicio", "2023-01-01", "Fecha inicio (YYYY-MM-DD)")
dbutils.widgets.text("fecha_fin", "2023-12-31", "Fecha fin (YYYY-MM-DD)")
dbutils.widgets.text("directorio_salida", "/Volumes/workspace/raw_seismic/usgs_quakes", "Directorio de salida")
dbutils.widgets.text("endpoint_api", "https://earthquake.usgs.gov/fdsnws/event/1/query", "Endpoint API")
dbutils.widgets.text("formato_respuesta", "geojson", "Formato de respuesta")
dbutils.widgets.text("tiempo_espera", "30", "Tiempo de espera (s)")

In [0]:
# Leer valores de los widgets
fecha_inicio = pendulum.parse(dbutils.widgets.get("fecha_inicio")).date()
fecha_fin = pendulum.parse(dbutils.widgets.get("fecha_fin")).date()
directorio_salida = Path(dbutils.widgets.get("directorio_salida"))
url_endpoint = dbutils.widgets.get("endpoint_api")
URL_API = url_endpoint
formato_respuesta = dbutils.widgets.get("formato_respuesta")
tiempo_espera = int(dbutils.widgets.get("tiempo_espera"))

In [0]:
archivos_guardados = []
fecha_actual = fecha_inicio

sesion = requests.Session()

while fecha_actual <= fecha_fin:
    # USGS necesita rango [starttime, endtime); usamos el día siguiente como límite superior
    fecha_str_inicio = fecha_actual.to_date_string()
    fecha_str_fin = fecha_actual.add(days=1).to_date_string()

    directorio_salida_diario = (
        directorio_salida
        / f"{fecha_actual.year:04d}"
        / f"{fecha_actual.month:02d}"
        / f"{fecha_actual.day:02d}"
    )
    directorio_salida_diario.mkdir(parents=True, exist_ok=True)

    ruta_archivo = directorio_salida_diario / "usgs_earthquakes.json"

    parametros = {
        "format": formato_respuesta,
        "starttime": fecha_str_inicio,
        "endtime": fecha_str_fin,
    }

    try:
        respuesta = sesion.get(URL_API, params=parametros, timeout=tiempo_espera)
        respuesta.raise_for_status()
    except requests.RequestException:
        fecha_actual = fecha_actual.add(days=1)
        continue

    # Guardar bytes del JSON tal cual lo devuelve la API
    ruta_archivo.write_bytes(respuesta.content)

    archivos_guardados.append(str(ruta_archivo))
    fecha_actual = fecha_actual.add(days=1)

print(f"Archivos guardados: {len(archivos_guardados)}")
